# Base ESM2 LLR vs Fitness — per-domain Spearman

Computes Spearman ρ between a base-model LLR matrix and the measured fitness
values for every protein domain.

The LLR dictionary is assumed to be in memory as `llr_dict`, keyed by
**UniProt accession** (e.g. `P00519`), with each entry containing:

- `sequence` — the (possibly truncated) full protein sequence used for scoring
- `pfam`     — the Pfam family of the domain (e.g. `PF00018`)
- `LLR`      — array of shape `(seq_len, 20)` with amino acids in
  `FITNESS_AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"` (same order as the fitness
  matrix), so no per-AA reindexing is needed.

The fitness dictionary is loaded from
`gs://domainome-data/ESM2/dict_fitness.pkl.gz` (same source as in
`ESM_single_domain_llr_compare_ns_rs.ipynb`) and is keyed by **`domain_id`**
(e.g. `P19878_PF00018_241`). The UniProt prefix is used to join into
`llr_dict`, and `dom_seq` is located inside the LLR sequence to align
positions.

Final output: a `pandas.DataFrame` `base_spearman_df` with columns
`domain_id` and `PG2_spearman`.

In [1]:
import os
import io
import gzip
import pickle
import warnings
import logging

import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.stats import spearmanr
from dotenv import load_dotenv
from google.cloud import storage

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

/var/folders/30/5r4_75_n4bz34nyphw8tfv3m0000gn/T/ipykernel_5859/3036034579.py:12: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.stats import spearmanr


In [2]:
# Amino-acid order shared by the new LLR matrices and the fitness matrices.
FITNESS_AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

GCS_PROJECT          = "plm-study-484223"
GCS_BUCKET           = "domainome-data"
GCS_DOMAIN_DATA_BLOB = "ESM2/dict_fitness.pkl.gz"

MIN_PAIRS = 10  # skip domains with fewer than this many (pos, mut) fitness measurements

In [3]:
# GCS client — fitness dict is streamed straight into memory, never written to disk.
load_dotenv()
cred_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if cred_path:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cred_path

_gcs_client = storage.Client()
_gcs_bucket = _gcs_client.bucket(GCS_BUCKET)


def gcs_load_pickle_gz(blob_path):
    blob = _gcs_bucket.blob(blob_path)
    data = blob.download_as_bytes()
    with gzip.GzipFile(fileobj=io.BytesIO(data), mode="rb") as gz:
        return pickle.load(gz)

In [4]:
bucket = _gcs_client.bucket("domainome-data")
blob = bucket.blob(
    "base_LLRs.pkl.gz"
)

blob.download_to_filename("base_LLRs.pkl.gz")

with gzip.open("base_LLRs.pkl.gz", "rb") as f:
    llr_dict = pickle.load(f)

In [5]:
try:
    fitness_dict
except NameError:
    print(f"Loading fitness dict from gs://{GCS_BUCKET}/{GCS_DOMAIN_DATA_BLOB}")
    fitness_dict = gcs_load_pickle_gz(GCS_DOMAIN_DATA_BLOB)

print(f"Loaded {len(fitness_dict)} fitness entries")
print(f"`llr_dict` already in memory: {len(llr_dict)} UniProt entries")

Loading fitness dict from gs://domainome-data/ESM2/dict_fitness.pkl.gz
Loaded 522 fitness entries
`llr_dict` already in memory: 428 UniProt entries


## Per-domain Spearman ρ

For each `domain_id` in `fitness_dict`:

1. Strip the UniProt prefix (everything before the first `_`) to look up the
   matching entry in `llr_dict`.
2. Locate `dom_seq` inside `llr_dict[uniprot]["sequence"]` so the domain's
   first residue maps to a column of the LLR matrix.
3. For every measured `(position, mut_aa)` cell in the fitness matrix that
   isn't `NaN` and isn't the wild-type residue, pair `fitness[i, j]` with
   `LLR[idx_in_seq + i, j]` (same AA order on both sides).
4. Compute Spearman ρ if there are at least `MIN_PAIRS` valid pairs.

In [6]:
def spearman_for_domain(domain_id, fd, llr_entry, min_pairs=MIN_PAIRS):
    """Return Spearman ρ between the base LLR and measured fitness, or None."""
    fitness = fd.get("fitness")
    dom_seq = fd.get("dom_seq")
    if fitness is None or dom_seq is None:
        return None

    sequence = llr_entry["sequence"]
    llr      = np.asarray(llr_entry["LLR"])  # shape (seq_len, 20)

    # The LLR is scored on `sequence` (which may be a truncated window of the
    # full protein). Find where the domain sits inside that window.
    idx_in_seq = sequence.find(dom_seq)
    if idx_in_seq < 0:
        return None

    n_pos = llr.shape[0]
    fits, llr_vals = [], []

    for i, wt_aa in enumerate(dom_seq):
        col = idx_in_seq + i
        if col >= n_pos or i >= len(fitness):
            break
        row = fitness[i]
        for j, mut_aa in enumerate(FITNESS_AA_ORDER):
            if j >= len(row):
                continue
            val = row[j]
            if val is None or mut_aa == wt_aa:
                continue
            try:
                fval = float(val)
            except (TypeError, ValueError):
                continue
            if np.isnan(fval):
                continue
            fits.append(fval)
            llr_vals.append(float(llr[col, j]))

    if len(fits) < min_pairs:
        return None

    rho, _ = spearmanr(np.asarray(fits), np.asarray(llr_vals))
    if np.isnan(rho):
        return None
    return float(rho)

In [7]:
rows = []
skipped_no_llr   = 0
skipped_no_match = 0
skipped_too_few  = 0

for domain_id, fd in tqdm(fitness_dict.items(), desc="Spearman per domain"):
    if not isinstance(fd, dict):
        continue

    uniprot_id = domain_id.split("_", 1)[0]
    llr_entry  = llr_dict.get(uniprot_id)
    if llr_entry is None:
        skipped_no_llr += 1
        continue

    rho = spearman_for_domain(domain_id, fd, llr_entry)
    if rho is None:
        # distinguish reasons for diagnostics
        sequence = llr_entry["sequence"]
        if fd.get("dom_seq") is None or sequence.find(fd["dom_seq"]) < 0:
            skipped_no_match += 1
        else:
            skipped_too_few += 1
        continue

    rows.append({"domain_id": domain_id, "PG2_spearman": rho})

base_spearman_df = pd.DataFrame(rows, columns=["domain_id", "PG2_spearman"])

print(f"Computed Spearman for {len(base_spearman_df)} domains "
      f"(skipped {skipped_no_llr} no-LLR, "
      f"{skipped_no_match} no-match, {skipped_too_few} too-few)")
if len(base_spearman_df):
    print(f"Mean ρ   : {base_spearman_df['PG2_spearman'].mean():+.4f}")
    print(f"Median ρ : {base_spearman_df['PG2_spearman'].median():+.4f}")

Spearman per domain: 100%|██████████| 522/522 [00:00<00:00, 1359.41it/s]

Computed Spearman for 471 domains (skipped 6 no-LLR, 0 no-match, 45 too-few)
Mean ρ   : +0.3084
Median ρ : +0.3108


In [9]:
base_spearman_df.shape

(471, 2)

In [10]:
base_spearman_df.head()

,domain_id,PG2_spearman
0,A0PJY2_PF00096_289,0.064622
1,A1X283_PF00018_155,0.407009
2,A1X283_PF00018_222,0.286978
3,A6NK59_PF07525_532,0.023704
4,O00308_PF00397_440,0.442646


In [11]:
bucket = _gcs_client.bucket("domainome-data")
blob = bucket.blob(
    "ESM2/base_ESM2_LLRs.pkl.gz"
)

blob.download_to_filename("base_LLRs.pkl.gz")

with gzip.open("base_LLRs.pkl.gz", "rb") as f:
    esm2_llr_dict = pickle.load(f)

In [12]:
esm2_llr_dict

{'P00519': {'sequence': 'MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTVYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQGVRGAVSTLLQAPELPTKTRTSRRAAEHRDTTDVPEMPHSKGQGESDPLDHEPAVSPLLPRKERGPPEGGLNEDERLLPKDKKTNLFSALIKKKKKTAPTPPKRSSSFREMDGQPERRGAGEEEGRDISNGALAFTPLDTADPAKSPKPSNGAGVPNGALRESGGSGFRSPHLWKKSSTLTSSRLATGEEEGGGSSSKRFLRSCSASCVPHGAKDTEWRSVTLPRDLQSTGRQFDSSTFGGHKSEKPALPRKRAGENRSDQVTRGTVTPPPRLVKKNEEAADEVFKDIMESSPGSSPPNLTPKPLRRQVTVAPASGLPHKEEAGKGSALGTPAAAEPVTPTSKAGSGAPGGTSKGPAEESRVRRHKHSSESPGRDKGKLSRLKPAPPPPPAASAGKAGGKPSQSPSQEAAGEAVLGAKTKATSLVDAVNSDAAKPSQPGEGLKKPVLPATPKPQSAKPSGTP